# Práctica 3 · UrbanIng-V2X

**Objetivo:** detectar vehículos, generar sus bounding boxes 3D y contar los presentes en la intersección.

Se utilizan únicamente los LiDAR de infraestructura **11, 12, 31 y 32**. **Sin tracking:** cada fotograma se procesa de forma independiente.

## 1. Datos

Secuencias del cruce `crossing1`:

- `20241126_0024_crossing1_09`
- `20241126_0008_crossing1_01`
- `20241127_0000_crossing1_00`

Los sensores 11 y 12 comparten ubicación, pero cada LiDAR tiene su propia calibración. No se utilizan LiDAR de vehículos ni cámaras.

In [ ]:
from pathlib import Path
import csv, json, html
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from main import SEQUENCES, SENSORS, load_frame, transform, voxelize, ground_plane
from detector import detect_improved, local_ground, DEFAULTS
from road import load_road, load_lane_segments, inside, draw_road, box_polygon

ROOT = Path.cwd()
assert (ROOT / 'main.py').is_file(), 'Para reejecutar, abrir el notebook desde la carpeta del proyecto.'
plt.rcParams.update({'figure.dpi': 115, 'font.size': 10, 'axes.titlesize': 12})

def tabla(headers, rows):
    header = ''.join('<th style="text-align:left;padding:8px">'+html.escape(str(v))+'</th>' for v in headers)
    body = ''.join('<tr>'+''.join('<td style="padding:8px;border-bottom:1px solid #ddd">'+html.escape(str(v))+'</td>' for v in row)+'</tr>' for row in rows)
    display(HTML('<table style="border-collapse:collapse"><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))

print('Sensores utilizados:', ', '.join(SENSORS))
print('Asociación entre fotogramas: desactivada (sin tracking).')

## 2. Sincronización y fusión de los cuatro LiDAR

Se seleccionan los archivos del mismo instante mediante `timesync_info.csv` y se transforma cada nube a coordenadas globales con su calibración: **`p_global = R · p_LiDAR + t`**.

Las cuatro nubes se fusionan antes de detectar, para evitar sumar detecciones independientes del mismo vehículo. Se muestra el fotograma 100 de `20241126_0008_crossing1_01`.

In [ ]:
sequence_name = SEQUENCES[1]
frame_index = 100
folder = ROOT / 'data' / 'dataset' / sequence_name
with (folder / 'timesync_info.csv').open(newline='') as stream:
    sync = {row[0]: row[1:] for row in csv.reader(stream)}
calibration = json.loads((folder / 'calibration.json').read_text())
points, sources = load_frame(folder, calibration, sync, frame_index)
timestamp = int(sync['timestamp_ms'][frame_index])
roi = [-45, 45, -45, 45]
road = load_road(ROOT / 'data/crossings_lanelet2map.osm', roi)
lanes = load_lane_segments(ROOT / 'data/crossings_lanelet2map.osm')
config = {**DEFAULTS, **json.loads((ROOT / 'detector_config.json').read_text())['parameters']}

tabla(['Sensor', 'Archivo sincronizado'], sources.items())
print(f'Fotograma {frame_index} · timestamp {timestamp} ms · {len(points):,} puntos fusionados')
fig, ax = plt.subplots(figsize=(10, 8))
for sensor in SENSORS:
    with np.load(folder / sensor / sources[sensor], allow_pickle=False) as data:
        xyz = np.column_stack([data[k] for k in ('x', 'y', 'z')])
    xyz = transform(xyz[np.isfinite(xyz).all(axis=1)], calibration[sensor]['extrinsics']['gTl'])
    xyz = xyz[(np.abs(xyz[:, 0]) <= 45) & (np.abs(xyz[:, 1]) <= 45)]
    xyz = voxelize(xyz, .45)
    ax.scatter(xyz[:, 0], xyz[:, 1], s=.7, alpha=.6, label=sensor)
ax.set(xlim=roi[:2], ylim=roi[2:], aspect='equal', xlabel='X global (m)', ylabel='Y global (m)', title='Cuatro puntos de vista en un sistema común')
ax.legend(markerscale=5, fontsize=8, loc='upper left')
fig.tight_layout(); display(fig); plt.close(fig)

## 3. Filtrado de la nube

Se reduce la densidad de puntos con vóxeles, se estima la altura del suelo y se conservan los puntos elevados próximos a la calzada. El mapa vial delimita la zona de detección.

Los vehículos detenidos también se consideran: no se exige movimiento.

In [ ]:
cropped = points[(np.abs(points[:, 0]) <= 45) & (np.abs(points[:, 1]) <= 45)]
reduced = voxelize(cropped, .15)
plane = ground_plane(reduced)
height, ground_model = local_ground(reduced, plane)
foreground = reduced[(height >= config['min_height']) & (height <= 4.5) & inside(reduced, road.buffer(.5))]
tabla(['Etapa', 'Puntos'], [('Dentro de la ROI', len(cropped)), ('Tras vóxeles', len(reduced)), ('Elevados cerca de calzada', len(foreground))])
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, cloud, title in zip(axes, [reduced, foreground], ['Nube reducida y máscara vial', 'Puntos para el agrupamiento']):
    sample = cloud[::max(1, len(cloud)//50000)]
    ax.scatter(sample[:, 0], sample[:, 1], s=.6, color='#536878')
    draw_road(ax, road)
    ax.set(xlim=roi[:2], ylim=roi[2:], aspect='equal', xlabel='X (m)', ylabel='Y (m)', title=title)
fig.tight_layout(); display(fig); plt.close(fig)

## 4. Detección y bounding boxes

Se agrupan los puntos mediante DBSCAN y se unen fragmentos compatibles de un mismo vehículo. La unión debe cumplir restricciones de orientación, distancia y dimensiones.

Se ajustan cajas orientadas con base en el suelo local. Para completar observaciones parciales se comparan varias posiciones y orientaciones, conservando los puntos observados y penalizando invadir otras cajas. Se suprimen duplicados restantes.

**El conteo es el número de cajas aceptadas en el fotograma.** No se asocian vehículos entre instantes.

In [ ]:
processed, boxes, plane = detect_improved(points, roi, .15, road, config, lanes)
print('Parámetros de agrupamiento:', {k: config[k] for k in ('eps', 'min_points')})
print(f'Conteo estimado en este fotograma: {len(boxes)} candidatos a vehículo')
tabla(['Caja local', 'Centro XYZ (m)', 'Largo × ancho × alto (m)', 'Giro (rad)'],
      [(i+1, ', '.join(f'{v:.2f}' for v in b['center']), ' × '.join(f'{v:.2f}' for v in b['dimensions']), f"{b['yaw_rad']:.2f}") for i,b in enumerate(boxes[:6])])
print('Se muestran las primeras seis cajas; los índices no son identidades de tracking.')
fig, ax = plt.subplots(figsize=(10, 8))
sample = processed[::max(1, len(processed)//50000)]
ax.scatter(sample[:,0], sample[:,1], s=.5, color='0.7')
draw_road(ax, road)
for i,b in enumerate(boxes):
    ax.plot(*box_polygon(b).exterior.xy, color='crimson', linewidth=1.1)
    ax.text(*b['center'][:2], str(i+1), fontsize=7)
ax.set(xlim=roi[:2], ylim=roi[2:], aspect='equal', xlabel='X (m)', ylabel='Y (m)', title=f'Detección final: {len(boxes)} cajas · fotograma {frame_index}')
fig.tight_layout(); display(fig); plt.close(fig)

### Vídeos de las tres secuencias

Cada vídeo muestra 200 fotogramas a 10 fps. Se conserva el fondo original: suelo gris y puntos elevados azul claro. A la derecha, los puntos de todos los clústeres aceptados y sus cajas aparecen en el mismo naranja. El conteo corresponde a las cajas de cada instante, **sin tracking**. Se pueden pausar y reproducir a menor velocidad.

In [ ]:
from video_gallery import gallery
from IPython.display import HTML, display
display(HTML(gallery(ROOT, embed=True)))

## 5. Conteo en las tres secuencias

Conteos por fotograma obtenidos con los cuatro LiDAR de infraestructura. Cada punto de las gráficas es una detección independiente, **sin tracking**.

Son conteos estimados: pueden persistir falsas detecciones y omisiones. No se suman para obtener vehículos únicos.

In [ ]:
series = {}
for name in SEQUENCES:
    path = ROOT / 'results/improved' / name / 'detections.jsonl'
    records = [json.loads(line) for line in path.read_text().splitlines()]
    assert len(records) == 200, f'Se esperan 200 fotogramas en {name}'
    assert all(r['count'] == len(r['boxes']) for r in records)
    series[name] = records

tabla(['Secuencia', 'Fotogramas procesados', 'Vehículos detectados en el fotograma 100'],
      [(name, len(records), next(r['count'] for r in records if r['frame_index'] == 100))
       for name, records in series.items()])

fig, axes = plt.subplots(3, 1, figsize=(12, 8), layout='constrained')
for ax, (name, records) in zip(axes, series.items()):
    start = records[0]['timestamp_ms']
    times = [(r['timestamp_ms'] - start) / 1000 for r in records]
    ax.plot(times, [r['count'] for r in records], color='#146b9b', linewidth=1.5)
    ax.set(title=name, xlabel='Tiempo (s)', ylabel='Vehículos detectados', ylim=(0, None))
    ax.grid(alpha=.2)
display(fig); plt.close(fig)